In [2]:
import datasets

print("datasets version:", datasets.__version__)

datasets version: 3.6.0


In [ ]:
# Load the Banking77 dataset from Hugging Face's datasets library
from datasets import load_dataset

dataset = load_dataset(
    "PolyAI/banking77",
    trust_remote_code=True
    )

print(dataset)

C:\Users\Avinash Kr Barnwal\.cache\huggingface\modules\datasets_modules\datasets\PolyAI--banking77\17ffc2ed47c2ed928bee64127ff1dbc97204cb974c2f980becae7c864007aed9\banking77.py:23: SyntaxWarning: invalid escape sequence '\~'
  _CITATION = """\


Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})


In [4]:
print("Available splits:", list(dataset.keys()))

print("Training samples:", len(dataset["train"]))
print("Test samples:", len(dataset["test"]))

print("\nDataset features:")
print(dataset["train"].features)

print("\nFirst customer query:")
print(dataset["train"][0])

Available splits: ['train', 'test']
Training samples: 10003
Test samples: 3080

Dataset features:
{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer', 'direct_debit_payment_not_recognised', 'disposable_card_limits', 'edit_personal_details', 'exchange_charge', 'exchange_rate', 'exchange_via_app', 'ext

In [ ]:
# Analyse the distribution of intent labels across Banking77's training and test splits.

import pandas as pd

train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_names = dataset["train"].features["label"].names

def get_intent_distribution(df, split_name):
    counts = df["label"].value_counts().sort_index()

    return pd.DataFrame({
        "label_id": range(len(label_names)),
        "intent": label_names,
        f"{split_name}_count": [
            int(counts.get(i, 0))
            for i in range(len(label_names))
        ]
    })

train_distribution = get_intent_distribution(
    train_df, "train"
)
test_distribution = get_intent_distribution(
    test_df, "test"
)

distribution = train_distribution.merge(
    test_distribution,
    on=["label_id", "intent"]
)

print("Number of unique training intents:", train_df["label"].nunique())
print("Number of unique test intents:", test_df["label"].nunique())

print("\nTraining examples per intent:")
print(distribution["train_count"].describe())

print("\nTest examples per intent:")
print(distribution["test_count"].describe())

print("\nIntent distribution:")
display(distribution)

Number of unique training intents: 77
Number of unique test intents: 77

Training examples per intent:
count     77.000000
mean     129.909091
std       32.942207
min       35.000000
25%      112.000000
50%      127.000000
75%      159.000000
max      187.000000
Name: train_count, dtype: float64

Test examples per intent:
count    77.0
mean     40.0
std       0.0
min      40.0
25%      40.0
50%      40.0
75%      40.0
max      40.0
Name: test_count, dtype: float64

Intent distribution:


,label_id,intent,train_count,test_count
0,0,activate_my_card,159,40
1,1,age_limit,110,40
2,2,apple_pay_or_google_pay,126,40
3,3,atm_support,87,40
4,4,automatic_top_up,127,40
...,...,...,...,...
72,72,virtual_card_not_working,41,40
73,73,visa_or_mastercard,135,40
74,74,why_verify_identity,121,40
75,75,wrong_amount_of_cash_received,180,40


In [3]:
# Verify the saved evaluation CSV and check its structure and data quality.
import pandas as pd

# Load the CSV from the project root when the notebook runs inside notebooks/.
evaluation_df = pd.read_csv("../data/evaluation_sample.csv")
print("Shape:", evaluation_df.shape)
print("Columns:", evaluation_df.columns.tolist())
print("Missing values:\n", evaluation_df.isnull().sum())
print("Unique intents:", evaluation_df["intent"].nunique())
print(
    "Samples per intent:\n",
    evaluation_df["intent"].value_counts().value_counts().sort_index()
)

# Display a few records for a quick visual inspection.
display(evaluation_df.head())

Shape: (231, 5)
Columns: ['sample_id', 'source_index', 'text', 'label', 'intent']
Missing values:
 sample_id       0
source_index    0
text            0
label           0
intent          0
dtype: int64
Unique intents: 77
Samples per intent:
 count
3    77
Name: count, dtype: int64


,sample_id,source_index,text,label,intent
0,test_02855,2855,I need help activating my new card.,0,activate_my_card
1,test_02856,2856,how to activate card?,0,activate_my_card
2,test_02859,2859,"I just got my card and want to use it, how do ...",0,activate_my_card
3,test_00480,480,how old do i need to be to get an account for ...,1,age_limit
4,test_00502,502,What's the minimum age for opening an account?,1,age_limit
